#### Imports

In [12]:
import json
import pandas as pd
import random
import string

#### Parse jsonl data into something we can use

In [5]:
comments_input: str = "./data/r_k12sysadmin_comments.jsonl"
posts_input: str = "./data/r_k12sysadmin_posts.jsonl"

post_data = []
comment_data = []

with open(posts_input, "r") as f:
    for line in f:
        line = line.strip()
        if line:
            post_data.append(json.loads(line))

# now data has a list of json objects that hold the posts


#### Format the data

Now that I have the json object, I want to do a few things before filtering.
1. Convert it into a dataframe, as I know how to work with those better.
2. That dataframe should have time, anonymized name, subreddit, post id, and post content.
3. Make another dataframe that maps the generated id to the username. This is just for testing and will not be included.

In other words, I should have one dataframe that has all the data from the comments and another that generates and stores unique ids for all the users.

In [10]:
def generate_random_string(length: int) -> str:
    characters = string.ascii_letters + string.digits
    random_string = ''.join(random.choice(characters) for _ in range(length))
    return random_string

In [26]:
posts: pd.DataFrame = pd.DataFrame(columns=["author", "time_of_post", "subreddit", "post_id", "post_link", "title", "content"])
user_ids: pd.DataFrame = pd.DataFrame(columns=["username", "id"])

for post in post_data:
    # get all the data we need for each post
    author: str = post["author"]
    time_of_post: int = post["created"]
    subreddit: str = post["subreddit"]
    post_id: str = post["id"]
    post_link: str = post["url"]
    title: str = post["title"]
    content: str = post["selftext"]

    if user_ids[user_ids["username"] == author].empty:
        random_id: str = generate_random_string(10)
        new_data: pd.DataFrame = pd.DataFrame([{"username": author, "id": random_id}])
        user_ids = pd.concat([user_ids, new_data], ignore_index=True)

    author = str(user_ids[user_ids["username"] == author]["id"].values[0])

    new_post: pd.DataFrame = pd.DataFrame([{"author": author, "time_of_post": time_of_post, "subreddit": subreddit, "post_id": post_id, "post_link": post_link, "title": title, "content": content}])
    posts = pd.concat([posts, new_post], ignore_index=True)
    print(content)

posts.to_csv("processed.csv")

The secure browser we use for testing has notified us that in order to continue using the browser, we must ensure that CrOS is on the LTS channel and NativeClientForceAllowed is enabled. No issues on the latter, but I'm a little unsure on moving to LTS. 

Currently we just let Chromebooks upgrade on the normal channel as needed. According to the information given to me, NaCl is will cease to work after updating to CrOS 133. My concern is waiting until the release of the next LTS version and what version that will be. I can't find much information available on upcoming releases, aside from this [Chromium Dash site](https://chromiumdash.appspot.com/schedule). If I'm looking at it right, 132 LTC will be available Jan 21 and 132 LTS will follow in April. 

I'm just looking for some advice on making the switch without causing too much headache. I feel like the best option is to pin 131 until 132 LTC becomes available, then move to LTS in the Spring, but it feels like that's cutting it close

#### Filter the data

Now the data is in a usable format, I now need to filter by keyword. I'll search the content and title columns and see if either of them contain a keyword. I must make sure that they are case insensitive as well.

In [27]:
filtered_posts: pd.DataFrame = pd.DataFrame(columns=["author", "time_of_post", "subreddit", "post_id", "post_link", "title", "content"])

keywords: list[str] = ["powerschool"]

pattern = '|'.join(keywords)

mask = (posts["title"].str.lower().str.contains(pattern, case=False, na=False) |
        posts["content"].str.lower().str.contains(pattern, case=False, na=False))
filtered_posts = posts[mask]

filtered_posts.to_csv("filtered.csv")